In [1]:
!pip install py-boost cupy-cuda12x

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.7/198.7 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: treelite
    Found existing installation: treelite 4.6.1
    Uninstalling treelite-4.6.1:
      Successfully uninstalled treelite-4.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires treelite<5.0.0,>=4.6.1, but you have treelite 3.9.1 which is incompatible.


In [2]:
!pip install iterative-stratification

In [3]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

#from utils import SEED, DATA_DIR, N_FOLDS, compute_macro_auc

SEED = 1234
DATA_DIR = "Data/"
N_FOLDS = 5

FEATURES_DIR = Path("features")
CHECKPOINT_DIR = Path("checkpoints_pyboost")

# Optuna-tuned params (25 trials on fold 0)
PARAMS = dict(
    ntrees=5000,
    lr=0.0566,
    max_depth=7,
    min_data_in_leaf=88,
    lambda_l2=2.076,
    subsample=0.88,
    colsample=0.76,
    max_bin=64,
    es=200,
    verbose=100,
    gd_steps=1,
    use_hess=False,
)

In [4]:
def compute_macro_auc(y_true, y_pred, target_cols):
    """Compute macro ROC-AUC with per-target breakdown."""
    aucs = {}
    for i, col in enumerate(target_cols):
        y_t = y_true[:, i]
        if y_t.sum() >= 2 and (len(y_t) - y_t.sum()) >= 2:
            aucs[col] = roc_auc_score(y_t, y_pred[:, i])
    return float(np.mean(list(aucs.values()))), aucs

In [5]:
def verify_submission(submit, sample):
    """Assert submission matches sample format."""
    assert submit.shape == sample.shape, f"Shape mismatch: {submit.shape} vs {sample.shape}"
    assert submit.columns == sample.columns, "Column mismatch"
    for col in submit.columns:
        assert submit[col].dtype == sample[col].dtype, f"Dtype mismatch for {col}"
    print(f"  Format verified: {submit.shape[0]:,} rows, {submit.shape[1]} cols")

In [6]:
t0 = time.time()
print("=" * 60)
print("Step 4: Train PyBoost (SketchBoost)")
print("=" * 60)


Step 4: Train PyBoost (SketchBoost)


In [7]:
try:
  import torch
  if not torch.cuda.is_available():
      print("\nERROR: PyBoost requires NVIDIA GPU with CUDA.")
      print("  This script uses cupy + py-boost which only work on CUDA GPUs.")
      print("  Skipping PyBoost training.")
except ImportError:
      print("\nERROR: torch not found. Install PyTorch with CUDA support.")


In [8]:
try:
    import cupy as cp
    from py_boost import SketchBoost
except ImportError:
        print("\nERROR: cupy and/or py-boost not installed.")
        print("  pip install py-boost cupy-cuda12x")

In [9]:
# 1. Load features
print("\n[1/4] Loading features...")
# with open(FEATURES_DIR / "meta.json") as f:
#     meta = json.load(f)
# target_cols = meta["target_cols"]

# train_feat = pl.read_parquet(FEATURES_DIR / "train_features.parquet")
# test_feat = pl.read_parquet(FEATURES_DIR / "test_features.parquet")
# train_tgt = pl.read_parquet(FEATURES_DIR / "targets.parquet")

train_feat = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1300/train_combined_1300.parquet')
test_feat = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1300/test_combined_1300.parquet')
train_tgt = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_target.parquet')

target_cols = train_tgt.drop('customer_id').columns


X_train = train_feat.drop("customer_id").to_numpy().astype(np.float32)
X_test = test_feat.drop("customer_id").to_numpy().astype(np.float32)
y = train_tgt.select(target_cols).to_numpy().astype(np.float32)
print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")


[1/4] Loading features...
  X_train: (750000, 1300), X_test: (250000, 1300)


In [10]:
    # 2. Check cache
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
cache_file = CHECKPOINT_DIR / "pyboost_predictions.npz"
if cache_file.exists():
  print(f"\n  Predictions exist at {cache_file}! Delete to retrain.")

In [11]:
 # 3. Train
print(f"\n[2/4] Training SketchBoost {N_FOLDS}-fold...")
mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros_like(y, dtype=np.float64)
test_preds = np.zeros((X_test.shape[0], len(target_cols)), dtype=np.float64)
fold_scores = []

for fold_idx, (tr_idx, val_idx) in enumerate(mskf.split(X_train, y)):
    t_fold = time.time()
    print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ──", flush=True)

    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = SketchBoost('bce', **PARAMS)
    model.fit(X_tr, y_tr, eval_sets=[{'X': X_val, 'y': y_val}])

    val_raw = model.predict(X_val)
    test_raw = model.predict(X_test)
    val_prob = 1.0 / (1.0 + np.exp(-val_raw))
    test_prob = 1.0 / (1.0 + np.exp(-test_raw))

    oof_preds[val_idx] = val_prob
    test_preds += test_prob / N_FOLDS

    fold_auc = roc_auc_score(y_val, val_prob, average='macro')
    fold_scores.append(fold_auc)
    print(f"  Fold {fold_idx+1} AUC={fold_auc:.5f} ({(time.time()-t_fold)/60:.1f} min)", flush=True)

    del model, X_tr, X_val, y_tr, y_val, val_raw, test_raw, val_prob, test_prob
    cp.get_default_memory_pool().free_all_blocks(); gc.collect()



[2/4] Training SketchBoost 5-fold...

  ── Fold 1/5 ──
[09:49:24] Stdout logging level is INFO.
[09:49:24] GDBT train starts. Max iter 5000, early stopping rounds 200
[09:49:32] Iter 0; Sample 0, BCE = 0.10427407277263313; 
[09:50:22] Iter 100; Sample 0, BCE = 0.08643530371191434; 
[09:51:17] Iter 200; Sample 0, BCE = 0.08456999007152075; 
[09:52:12] Iter 300; Sample 0, BCE = 0.08373451446051633; 
[09:53:07] Iter 400; Sample 0, BCE = 0.0832326906128256; 
[09:54:02] Iter 500; Sample 0, BCE = 0.08287520384800857; 
[09:54:57] Iter 600; Sample 0, BCE = 0.08266320579603745; 
[09:55:51] Iter 700; Sample 0, BCE = 0.08250528464268501; 
[09:56:46] Iter 800; Sample 0, BCE = 0.08236600487757822; 
[09:57:41] Iter 900; Sample 0, BCE = 0.08226371986786524; 
[09:58:36] Iter 1000; Sample 0, BCE = 0.08218008585805367; 
[09:59:30] Iter 1100; Sample 0, BCE = 0.08210739370994143; 
[10:00:25] Iter 1200; Sample 0, BCE = 0.08204648275622677; 
[10:01:20] Iter 1300; Sample 0, BCE = 0.08199488005420964; 
[10:0

In [12]:
    # 4. Results
macro_auc = float(np.mean([
    roc_auc_score(y[:, i], oof_preds[:, i]) for i in range(len(target_cols))
]))
print(f"\n[3/4] Results:")
print(f"  Fold AUCs: {['%.5f' % s for s in fold_scores]}")
print(f"  OOF Macro AUC: {macro_auc:.5f}")

# Save
np.savez_compressed(cache_file,
                        oof_preds=oof_preds, test_preds=test_preds,
                        fold_scores=fold_scores, macro_auc=macro_auc)
print(f"  Saved: {cache_file}")

# Save submission
print("\n[4/4] Saving submission...")
# from utils import verify_submission

sample = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/sample_submit.parquet')

predict_cols = [c.replace("target_", "predict_") for c in target_cols]
submit = pl.DataFrame({"customer_id": test_feat["customer_id"]}).hstack(
        pl.DataFrame(test_preds.astype(np.float64), schema=predict_cols)
    )
verify_submission(submit, sample)
Path("submissions").mkdir(exist_ok=True)
submit.write_parquet("submissions/pyboost.parquet")
print(f"  Saved: submissions/pyboost.parquet")

print(f"\nDone in {(time.time()-t0)/60:.1f} min. OOF={macro_auc:.5f}")


[3/4] Results:
  Fold AUCs: ['0.83686', '0.83557', '0.83616', '0.83891', '0.83746']
  OOF Macro AUC: 0.83690
  Saved: checkpoints_pyboost/pyboost_predictions.npz

[4/4] Saving submission...
  Format verified: 250,000 rows, 42 cols
  Saved: submissions/pyboost.parquet

Done in 187.4 min. OOF=0.83690
